# Download and process BAST traffic count data
Source: https://www.bast.de/BASt_2017/DE/Verkehrstechnik/Fachthemen/v2-verkehrszaehlung/zaehl_node.html

In [1]:
import os
import requests

import geopandas as gpd
import numpy as np
import pandas as pd

from zipfile import ZipFile
from bs4 import BeautifulSoup


### Functions for download 
Functions for download and unzip BAST hourly data

In [2]:
#actual downloading -> Author: Johannes Gensheimer
def DownloadURL(url, save_path):
    #function from: https://stackoverflow.com/questions/9419162/download-returned-zip-file-from-url/14260592
    
    chunk_size=128
    r = requests.get(url, stream=True)
    with open(save_path, 'wb') as fd:
        for chunk in r.iter_content(chunk_size=chunk_size):
            fd.write(chunk)

#call download and unzip
def DownloadHourData_BAST(CounterNumber, year, FolderSave):

    BAST_Link = 'https://www.bast.de/videos/' + str(year) + '/zst' + str(CounterNumber) + '.zip'
    FileName = FolderSave + 'zst' + str(CounterNumber) + '_' + str(year) + '.zip'
    #download zip
    DownloadURL(BAST_Link, FileName)
    
    #unzip file if downloading was successfull
    try:
        zf = ZipFile(FileName, 'r')
        zf.extractall(FolderSave)
        zf.close()
        #remove zip file
        os.system('rm ' + FileName)
        return True
    except:
        print('Error while unzipping file: ' + FileName)
        return False

def DownloadRawData_BAST(year, FolderSave, link):
    FileName = FolderSave + 'DZ_' + str(year) + '_Rohdaten.zip'
    #download zip
    DownloadURL(link, FileName)
    
    #unzip file if downloading was successfull
    try:
        zf = ZipFile(FileName, 'r')
        zf.extractall(FolderSave)
        zf.close()
        #remove zip file
        os.system('rm ' + FileName)
        #return True
    except:
        print('Error while unzipping file: ' + FileName)
        return False
    
    #unzip next level of zip files (for each month)
    for month in range(1, 13):
        FileName = FolderSave + 'DZ_' + str(year) + '_' + str(month).zfill(2) + '_Rohdaten.zip'
        try:
            zf = ZipFile(FileName, 'r')
            zf.extractall(FolderSave)
            zf.close()
            #remove zip file
            os.system('rm ' + FileName)
        except:
            print('Error while unzipping file: ' + FileName)
    return True


### Function to process download request 
DZ_Nr_unique: Number of counting stations <br>
years: years that should be downloaded <br>
path: path to temp directory where the downloaded data is temporary stored <br>
keepOrigData: Bool if downloaded raw data should be deleted: True: not deleted, False: deleted <br>

In [3]:
def Process(DZ_Nr_unique, years, path, keepOrigData):

    #sort years array ascending
    years.sort()
    print(years)
    
    #create temporary directory
    if not os.path.exists(path):
        os.system('mkdir ' + path)

    #total number of counting stations that should be processed
    total_count = len(DZ_Nr_unique)

    #some prints
    print('All counter stations: ', DZ_Nr_unique)
    print('In total # = ', total_count)
    count = 1
    
    # empty dataframe for all data
    df_all = pd.DataFrame()
    
    #loop over all counting stations
    for nr in range(0,len(DZ_Nr_unique)):
        
        DZ_Nr = DZ_Nr_unique[nr]
        print('Processing ' , count, ' of ', total_count)
        count = count + 1
        
        #loop over years
        for year in years:
            
            #filename in temp dir
            f = 'zst' + str(DZ_Nr) + '_' + str(year) + '.csv'
            try:
                #download data
                DownloadHourData_BAST(DZ_Nr, year, path) # if download was successfull
                #read data
                df = pd.read_csv(path+f, delimiter=';')
                df_all = pd.concat([df_all, df], axis = 0)
                    
                #remove downloaded file if keepOrigData not True
                if not keepOrigData:
                    os.system('rm ' + path + f)
            except Exception as e:
                print("Could not download data for:"+str(DZ_Nr_unique[nr])+"/"+str(year))

    return df_all

def Process_raw(years, temp_path):
    #Find all links to zip files for the specified years on the BAST website
    BAST_Link = 'https://www.bast.de/DE/Publikationen/Daten/Verkehrstechnik/DZ.html'
    response = requests.get(BAST_Link)
    soup = BeautifulSoup(response.text, 'html.parser')
    links = soup.find_all('a', href=lambda x: x and '.zip' in x and any(str(v) in x for v in years))

    for tag in links:
        link = tag.get('href', None)
        if link is not None:
            matched = [v for v in years if str(v) in link].pop()
            print(matched)
            DownloadRawData_BAST(matched, temp_path, link) # download and unzip raw data for the specified year
            print(link)

In [4]:
def delete_unused_locations(location_list, path):
    locations = ['BY' + str(location) for location in location_list]
    print(locations)
    # print(f"Deleting data of unused locations")
    for folder in os.listdir(path):
        if folder == '.DS_Store': continue
        new_path = os.path.join(path, folder)
        for file in os.listdir(new_path):
            temp_path = os.path.join(new_path, file)
            #print(os.listdir(temp_path))
            print(temp_path)
            print(os.path.isfile(temp_path))
            print("test")
            if os.path.isfile(temp_path) and not (str(file[:6]) in locations):
                print("delete")
                os.system('rm ' + temp_path)

In [5]:
def extract_append_raw_data(df_old, path, newFile='BAST_CountingStations_daily_new.csv'):
     
    for folder in os.listdir(path):
        if folder == '.DS_Store': continue
        print(f"Processing {folder}")
        new_path = os.path.join(path, folder + '/')
        for file in os.listdir(new_path):

            # Handle header rows individually, due to different number of elements
            df_header1 = pd.read_csv(new_path + file, encoding= 'unicode_escape', nrows=1, header=None, sep=r'\s+')
            df_header2 = pd.read_csv(new_path + file, encoding= 'unicode_escape', nrows=1, header=None, sep=r'\s+', skiprows=1)
            df_header3 = pd.read_csv(new_path + file, encoding= 'unicode_escape', nrows=1, header=None, sep=r'\s+', skiprows=2)

            # Copy Dataframe Schema
            df_new = pd.DataFrame().reindex_like(df_old).dropna()
            df_body = pd.read_csv(new_path + file, encoding= 'unicode_escape', skiprows=3, header=None,  sep=r'\s+')   
            
            # Drop faulty line (necessary because of measurements in march for some reason)
            df_body = df_body.drop(df_body[df_body[0].astype(str).str.contains('i', regex=False)].index)

        # Extract data from header
            df_new['Datum'] = df_body[0]
            df_new['Stunde'] = df_body[1].str.split(":").str[0].astype(int)
            df_new['KFZ_R1'] = df_body[2]
            df_new['Wotag'] = pd.to_datetime(df_new['Datum'].astype(str), format='%y%m%d').dt.weekday
            df_new['Wotag'] += 1 # dt.weekday specifies Monday=0 .. Sunday=6

            # Fill NaN's with 0
            df_new = df_new.fillna(0)

            # Extract necessary information
            df_new = df_new.assign(TKNR=df_header1.iloc[0, 0][1:5])
            df_new = df_new.assign(Zst = df_header1.iloc[0, 0][5:9])
            df_new = df_new.assign(Land = df_header1.iloc[0, 1])
            df_new = df_new.assign(Strklas = df_header1.iloc[0, 2])
            df_new = df_new.assign(Strnum = df_header1.iloc[0, 3])
            df_new = df_new.assign(Fahrtzw = 'n')

            # number of lanes in each direction
            lanes_per_direction = [int(df_header2.iloc[0, 0][1:]), int(df_header2.iloc[0, 1])]
            offset_date_time = 2
            offset_kfz_sv = 2 * (lanes_per_direction[0] + lanes_per_direction[1])

            # For some reason 1 station is different than all the other stations..
            if 'BY9223' in file:
                column = 'KFZ_R'

                for direction in range(2): # code only works for 2 directions
                    # KFZ and LKW (SV)
                    count_name = column + str(direction + 1)
                    check_name = 'K_' + column + str(direction + 1)
                    df_temp_count = df_body.iloc[:][offset_date_time + (direction * lanes_per_direction[0])].str.strip().str[:-1].astype(int)
                    df_temp_check = df_body.iloc[:][offset_date_time + (direction * lanes_per_direction[0])].str.strip().str[-1:]
                    for lane in range(1, lanes_per_direction[direction]):
                        df_temp_count += df_body.iloc[:][offset_date_time + (direction * lanes_per_direction[0]) + lane].str.strip().str[:-1].astype(int)

                    df_new[count_name] = df_temp_count
                    df_new[check_name] = df_temp_check
            else: 

                columns_kfz_lkw = ['KFZ_R', 'Lkw_R']
                columns_rest = ['Mot_R', 'Pkw_R', 'Lfw_R', 'PmA_R', 'Bus_R',
                                'LoA_R', 'Lzg_R', 'Sat_R', 'Son_R'] # attention, Lzg = LmA + Sat
                columns_calc = ['PLZ_R', 'Lzg_R'] # Lzg = LmA + Sat, PLZ = Mot + Pkw + Lfw

                for direction in range(2):
                    # KFZ and LKW (SV)
                    for idx, column in enumerate(columns_kfz_lkw):
                        count_name = column + str(direction + 1)
                        check_name = 'K_' + column + str(direction + 1)
                        df_temp_count = df_body.iloc[:][offset_date_time + idx + (direction * 2 * lanes_per_direction[0])].str.strip().str[:-1].astype(int)
                        df_temp_check = df_body.iloc[:][offset_date_time + idx + (direction * 2 * lanes_per_direction[0])].str.strip().str[-1:]
                        for lane in range(1, lanes_per_direction[direction]):
                            df_temp_count += df_body.iloc[:][offset_date_time + idx + (direction * 2 * lanes_per_direction[0]) + (2 * lane)].str.strip().str[:-1].astype(int)

                        df_new[count_name] = df_temp_count
                        df_new[check_name] = df_temp_check
                    # Other classes
                    for idx, column in enumerate(columns_rest):
                        count_name = column + str(direction + 1)
                        check_name = 'K_' + column + str(direction + 1)
                        df_temp_count = df_body.iloc[:][offset_date_time + offset_kfz_sv + idx + (direction * 9 * lanes_per_direction[0])].str.strip().str[:-1].astype(int)
                        df_temp_check = df_body.iloc[:][offset_date_time + offset_kfz_sv + idx + (direction * 9 * lanes_per_direction[0])].str.strip().str[-1:]
                        for lane in range(1, lanes_per_direction[direction]):
                            df_temp_count += df_body.iloc[:][offset_date_time + offset_kfz_sv + idx + (direction * 9 * lanes_per_direction[0]) + (9 * lane)].str.strip().str[:-1].astype(int)
                        df_new[count_name] = df_temp_count
                        df_new[check_name] = df_temp_check

                    # Lzg = LmA + Sat
                    df_new['Lzg_R' + str(direction + 1)] += df_new['Sat_R' + str(direction + 1)]
                    # PLZ = Mot + Pkw + Lfw
                    df_new['PLZ_R' + str(direction + 1)] = df_new['Mot_R' + str(direction + 1)] + df_new['Pkw_R' + str(direction + 1)] + df_new['Lfw_R' + str(direction + 1)]
                    df_new['K_PLZ_R' + str(direction + 1)] = df_new['K_Mot_R' + str(direction + 1)] # Not sure how to handle this  
            df_old = pd.concat([df_old, df_new])
        print(f"Completed {folder}")
    print(f"Writing to file {newFile}")
    df_old.to_csv(newFile)
    return df_old


# Run the functions

In [6]:
bast_locations = gpd.read_file('bast_locations_selected.gpkg')
station_ids = np.array(bast_locations['MST_ID'].unique()) # counting station number

years = np.array([2023]) # years of interst

# download processed datasets from BAST homepage
df_daily = Process(station_ids, years, path='temp/', keepOrigData=False)
df_daily.to_csv('BAST_CountingStations_daily_until2022.csv')

[2023]
All counter stations:  [9140 9141 9772 9773 9082 9003 9083 9115 9066 9063 9242 9064 9217 9006
 9987 9986 9985 9151 9043 9222 9106 9207 9775 9774 9215 9212 9213 9220
 9219 9218 9211 9155 9228 9244 9229 9221 9830 9820 9810 9320 9102 9988]
In total # =  42
Processing  1  of  42
Processing  2  of  42
Processing  3  of  42
Processing  4  of  42
Error while unzipping file: temp/zst9773_2023.zip
Could not download data for:9773/2023
Processing  5  of  42
Processing  6  of  42
Processing  7  of  42
Processing  8  of  42
Error while unzipping file: temp/zst9115_2023.zip
Could not download data for:9115/2023
Processing  9  of  42
Processing  10  of  42
Processing  11  of  42
Processing  12  of  42
Processing  13  of  42
Processing  14  of  42
Processing  15  of  42
Processing  16  of  42
Error while unzipping file: temp/zst9986_2023.zip
Could not download data for:9986/2023
Processing  17  of  42
Error while unzipping file: temp/zst9985_2023.zip
Could not download data for:9985/2023
Proce

In [7]:
# Raw data
print(station_ids)
# create temporary directory to save raw data for the raw data download  
temp_path = 'temp_raw/'
if not os.path.exists(temp_path):
    os.makedirs(temp_path)
# Download raw data from BAST can be performed manually from the follwowing link and unzipped into the temp_raw/ folder
# URL = 'https://www.bast.de/DE/Publikationen/Daten/Verkehrstechnik/DZ.html'
# the downloaded zip file contains zip files for each month
years_raw = [2024] # select the years for which raw data should be downloaded
Process_raw(years_raw, temp_path) # get links for raw data download for the specified years

# Remove stations not in DZ_Nr_arr
delete_unused_locations(station_ids, temp_path)
# extract and append raw data to previous df
extract_append_raw_data(df_daily, temp_path, 'BAST_CountingStations_daily_until2024.csv')

[9140 9141 9772 9773 9082 9003 9083 9115 9066 9063 9242 9064 9217 9006
 9987 9986 9985 9151 9043 9222 9106 9207 9775 9774 9215 9212 9213 9220
 9219 9218 9211 9155 9228 9244 9229 9221 9830 9820 9810 9320 9102 9988]
2024
https://files.bast.de/index.php/s/Dtb5EW44mXzjHnA/download/DZ_2024_Rohdaten.zip
['BY9140', 'BY9141', 'BY9772', 'BY9773', 'BY9082', 'BY9003', 'BY9083', 'BY9115', 'BY9066', 'BY9063', 'BY9242', 'BY9064', 'BY9217', 'BY9006', 'BY9987', 'BY9986', 'BY9985', 'BY9151', 'BY9043', 'BY9222', 'BY9106', 'BY9207', 'BY9775', 'BY9774', 'BY9215', 'BY9212', 'BY9213', 'BY9220', 'BY9219', 'BY9218', 'BY9211', 'BY9155', 'BY9228', 'BY9244', 'BY9229', 'BY9221', 'BY9830', 'BY9820', 'BY9810', 'BY9320', 'BY9102', 'BY9988']
temp_raw/DZ_2024_08_Rohdaten/BY9394.248
True
test
delete
temp_raw/DZ_2024_08_Rohdaten/NI3455.248
True
test
delete
temp_raw/DZ_2024_08_Rohdaten/HE6905.248
True
test
delete
temp_raw/DZ_2024_08_Rohdaten/HE6426.248
True
test
delete
temp_raw/DZ_2024_08_Rohdaten/NI3361.248
True
test
de

,TKNR,Zst,Land,Strklas,Strnum,Datum,Wotag,Fahrtzw,Stunde,KFZ_R1,...,Bus_R2,K_Bus_R2,LoA_R2,K_LoA_R2,Lzg_R2,K_Lzg_R2,Sat_R2,K_Sat_R2,Son_R2,K_Son_R2
0,7834,9140,9,A,8,230101,7,s,1,88,...,0,a,0,a,0,a,0,a,0,a
1,7834,9140,9,A,8,230101,7,s,2,259,...,0,a,0,a,0,a,0,a,0,a
2,7834,9140,9,A,8,230101,7,s,3,268,...,0,a,0,a,0,a,0,a,0,a
3,7834,9140,9,A,8,230101,7,s,4,160,...,0,a,0,a,0,a,0,a,0,a
4,7834,9140,9,A,8,230101,7,s,5,96,...,0,a,0,a,0,a,0,a,0,a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,7935,9083,9,A,8,241231,2,n,20,560,...,0,-,1,-,2,-,1,-,0,-
740,7935,9083,9,A,8,241231,2,n,21,353,...,0,-,0,-,1,-,1,-,0,-
741,7935,9083,9,A,8,241231,2,n,22,288,...,1,-,0,-,1,-,1,-,0,-
742,7935,9083,9,A,8,241231,2,n,23,293,...,1,-,0,-,1,-,1,-,0,-


In [8]:
df_test = pd.read_csv('BAST_CountingStations_daily_until2024.csv', index_col=0).sort_values('Datum')
df_test['Datum'].max()

np.int64(241231)

In [9]:
#df_test = pd.read_csv('BAST_CountingStations_daily_until2022.csv', index_col=0)

df_daily.tail()


,TKNR,Zst,Land,Strklas,Strnum,Datum,Wotag,Fahrtzw,Stunde,KFZ_R1,...,Bus_R2,K_Bus_R2,LoA_R2,K_LoA_R2,Lzg_R2,K_Lzg_R2,Sat_R2,K_Sat_R2,Son_R2,K_Son_R2
8755,7734,9988,9,B,471,231231,7,s,20,192,...,0,-,1,-,0,-,0,-,0,-
8756,7734,9988,9,B,471,231231,7,s,21,145,...,0,-,2,-,0,-,0,-,0,-
8757,7734,9988,9,B,471,231231,7,s,22,108,...,0,-,1,-,0,-,0,-,0,-
8758,7734,9988,9,B,471,231231,7,s,23,101,...,1,-,0,-,0,-,0,-,0,-
8759,7734,9988,9,B,471,231231,7,s,24,90,...,0,-,1,-,2,-,2,-,0,-
